# DUPLICATE SAMPLES

**Purpose** 
Duplicate samples assess analytical precision and reproducibility, rather than absolute 
accuracy.

**Data Characteristics**

- Paired observations from the same source 
- Derived features (e.g. difference, ratio, RPD) 
- Often heteroscedastic (precision varies with concentration)

**Typical Anomaly Patterns**

- Pairwise anomalies: unusually large difference within a duplicate pair 
- Temporal anomalies: rising RPD over time
- Conditional anomalies: failures only at low or high concentrations

**Interpretation**

Duplicate anomalies typically indicate: 
- Poor sample homogenization 
- Inconsistent preparation 
- Instrument instability

This QC type naturally lends itself to:
- Pairwise modeling 
- Error modeling 
- Conditional anomaly detection


PRECISION_STATUS - Rule-based classification. Explores whether unsupervised learning techniques can identify additional patterns or anomalies not captured by predefined rules.

## Duplicates versus Replicates.
Dion "Duplicates duplicate across all scheme whereas replicates are duplicated only within one scheme."
Anything "ignored" is done by a user (person)"


In [1]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest


In [2]:
df = pd.read_csv(
    "../data/raw/ResultSet.csv",
    dtype={"INSTRUMENT_ID": "string"}
)

df.head()


,ANALYTICAL_TYPE,STD_LOT_CODE,STD_CODE,JOB_CODE,NUMERIC_FINAL_VALUE,ANALYSED_DATE,SCHEME_CODE,ANALYTE_CODE,STANDARD_STATUS,PRECISION_STATUS,...,INTERNAL_MAX_WARNING_INCLUSIVE,LIM_REP_VALUE,STAT_DL_VALUE,LIM_REP_DUP_VALUE,STAT_DL_DUP_VALUE,INTERNAL_TARGET_VALUE,PARENT_NUMERIC_FINAL_VALUE,UNIT_CODE,SPECIFICATION_CODE,INSTRUMENT_ID
0,Standard,OREAS_905,OREAS_905,TSV_LB0016663011,35.927171,2021-11-22 16:37:52.000000,GE_IMS40Q12,PB,UpperFailure,NaN,...,Y,10,1.25,15.0,1.25,30.40,NaN,MG_KG,OREAS_905,<NA>
1,Standard,OREAS_905,OREAS_905,TSV_LB0016663011,55.639610,2021-11-22 16:37:52.000000,GE_IMS40Q12,PB,UpperFailure,NaN,...,Y,10,1.25,15.0,1.25,30.40,NaN,MG_KG,OREAS_905,<NA>
2,Replicate,Sample,Sample,TSV_LB0016663011,1492.268392,2021-11-22 16:37:52.000000,GE_IMS40Q12,PB,NaN,Warning,...,NaN,10,1.25,15.0,1.25,NaN,1385.198107,MG_KG,NaN,<NA>
3,Replicate,Sample,Sample,TSV_LB0016663011,18.525452,2021-11-22 16:37:33.000000,GE_IMS40Q12,NI,NaN,Pass,...,NaN,10,5.00,15.0,5.00,NaN,16.334782,MG_KG,NaN,<NA>
4,Standard,OREAS_905,OREAS_905,TSV_LB0016663011,11.742601,2021-11-22 16:37:33.000000,GE_IMS40Q12,NI,UpperWarning,NaN,...,Y,10,5.00,15.0,5.00,9.54,NaN,MG_KG,OREAS_905,<NA>


## Columns - Duplicates Only

**ANALYTICAL_TYPE == 'Duplicate'**

**STD_LOT_CODE == 'Sample'**

**STD_CODE == 'Sample'**

**JOB_CODE == 'TSV_LB000\*'**

Conforms to label format.

**NUMERIC_FINAL_VALUE**

The NUMERIC_FINAL_VALUE represents the measured concentration for duplicate sample analyses. Summary statistics indicate a highly skewed distribution:

The mean (≈ 60.0) is substantially higher than the median (≈ 6.54), suggesting the presence of extreme high-value outliers.
The maximum value (≈ 16,778) is several orders of magnitude larger than the upper quartile (75% ≈ 30.17), reinforcing that a small number of large observations are heavily influencing the distribution.

The standard deviation (≈ 315.1) is also very large relative to the median, further indicating high variability driven by outliers.
There is a presence of negative values (min ≈ -43.7).

Overall, the distribution is right-skewed - Precision varies with concentration?

**ANALYSED_DATE**

Datetime

**SCHEME_CODE == 'GE_ICP40Q12'**

**ANALYTE_CODE**

`TA, TE, MO, CE, S, ZR, V, W, Y, CR, BE, CD, BA, TH, SN, SE, SC, NI, NB, LI, LA, HF, U, ZN, TI, PB, RB, MN, MG, CO, SR, CA, BI, AL, FE, SB, CU, K, P, AG, AS`

**STANDARD_STATUS** (nan)

**PRECISION_STATUS**

`Warning, Pass, IgnoreFailure, Failure`

**INTERNAL_MIN_VALUE** (nan)

**INTERNAL_MAX_VALUE** (nan)

**INTERNAL_MIN_INCLUSIVE**

**INTERNAL_MAX_INCLUSIVE** (nan)

**INTERNAL_MAX_WARNING_VALUE** (nan)

**INTERNAL_MIN_WARNING_VALUE** (nan)

**INTERNAL_MIN_WARNING_INCLUSIVE** (nan)

**INTERNAL_MAX_WARNING_INCLUSIVE** (nan)

**LIM_REP_VALUE**

`10 or 15`

**STAT_DL_VALUE**

`0.025, 1.25, 2.5, 5.0, 7.5, 12.5, 25.0, 50.0`

**LIM_REP_DUP_VALUE**

`15.0`

**STAT_DL_DUP_VALUE**

`0.025, 0.25, 1.25, 2.5, 5, 7.5, 12.5, 25, 50`

**INTERNAL_TARGET_VALUE** (nan)


**PARENT_NUMERIC_FINAL_VALUE**

Parent measurement?

Summary stats:

The mean (≈ 60.25) is substantially higher than the median (≈ 6.76), indicating the presence of high-value outliers that are inflating the average.
The maximum value (≈ 16,881) is several orders greater than the upper quartile (75% ≈ 30.60), confirming a long right tail.
The standard deviation (≈ 315.99) is large relative to the median.
The minimum value (≈ -58.80)?
There are no missing values.

The distribution is right-skewed - Variability increases with concentration?

**UNIT_CODE**

`MG_KG, PERC`

**SPECIFICATION_CODE** (nan)

**INSTRUMENT_ID** (nan)


In [3]:
important_cols = [
    "NUMERIC_FINAL_VALUE",         # Duplicate measurement (primary observed value)
    "PARENT_NUMERIC_FINAL_VALUE",  # Original measurement (paired with duplicate)
    "ANALYSED_DATE",               # Temporal feature
    "ANALYTE_CODE",                # Element/compound being measured
    "STAT_DL_VALUE",               # Detection limit for original sample (affects low-end variability)
    "STAT_DL_DUP_VALUE",           # Detection limit for duplicate sample (may differ slightly)
    "LIM_REP_VALUE",               # Reporting limit for original sample (rounding/threshold effects)
    "LIM_REP_DUP_VALUE",           # Reporting limit for duplicate sample
    "UNIT_CODE",                   # Measurement unit (e.g. MG_KG)
    "PRECISION_STATUS"             # Rule-based QC classification (Pass/Warning/Failure)
]


In [4]:
# Filter duplicates AND select only important columns
df_duplicates = df[
    df["ANALYTICAL_TYPE"].str.strip().str.lower() == "duplicate"
][important_cols].copy()

# Export to CSV
df_duplicates.to_csv("../data/processed/duplicates_only.csv", index=False)


In [5]:
# Filter replicates AND select only important columns
df_replicates = df[
    df["ANALYTICAL_TYPE"].str.strip().str.lower() == "replicate"
][important_cols].copy()

# Export to CSV
df_replicates.to_csv("../data/processed/replicates_only.csv", index=False)


## Intermediate Model

Build model for both duplicates and replicates with the following derived columns added:
Add Absoulute difference
Mean of X1 and X2
RPD calculated from absolute difference and the mean of the duplciate/parent values.


In [11]:
def build_precision_features(df_input):
    """
    Build a modelling-ready dataset for QC precision analysis.

    Derives pairwise features between a QC sample (duplicate/replicate)
    and its parent sample:
        - ABS_DIFF   : Absolute difference
        - MEAN_CONC  : Mean concentration
        - RPD        : Relative Percent Difference (rounded to 2 d.p.)

    Notes:
        - Handles divide-by-zero using np.nan
        - Designed for duplicate and replicate samples
    """
    
    df_model = df_input.copy()

    # Absolute difference
    df_model["ABS_DIFF"] = abs(
        df_model["NUMERIC_FINAL_VALUE"] - df_model["PARENT_NUMERIC_FINAL_VALUE"]
    )

    # Mean concentration
    df_model["MEAN_CONC"] = (
        df_model["NUMERIC_FINAL_VALUE"] + df_model["PARENT_NUMERIC_FINAL_VALUE"]
    ) / 2

    # Relative Percent Difference (RPD)
    df_model["RPD"] = np.where(
        (df_model["NUMERIC_FINAL_VALUE"] + df_model["PARENT_NUMERIC_FINAL_VALUE"]) != 0,
        df_model["ABS_DIFF"] / df_model["MEAN_CONC"] * 100,
        np.nan
    )

    # Round to 2 decimal places (still float)
    df_model["RPD"] = df_model["RPD"].round(2)

    # Final column order
    df_model = df_model[
        [
            "ANALYTE_CODE",
            "ANALYSED_DATE",
            "NUMERIC_FINAL_VALUE",
            "PARENT_NUMERIC_FINAL_VALUE",
            "ABS_DIFF",
            "MEAN_CONC",
            "RPD",
            "PRECISION_STATUS",
            "STAT_DL_VALUE",
            "STAT_DL_DUP_VALUE",
            "LIM_REP_VALUE",
            "LIM_REP_DUP_VALUE",
            "UNIT_CODE"
        ]
    ]

    return df_model
    

In [12]:
df_duplicates_model = build_precision_features(df_duplicates)
df_replicates_model = build_precision_features(df_replicates)


In [13]:
df_duplicates_model.to_csv("../data/processed/duplicates_model.csv", index=False)
df_replicates_model.to_csv("../data/processed/replicates_model.csv", index=False)


In [14]:
df_duplicates_model.head(10)


,ANALYTE_CODE,ANALYSED_DATE,NUMERIC_FINAL_VALUE,PARENT_NUMERIC_FINAL_VALUE,ABS_DIFF,MEAN_CONC,RPD,PRECISION_STATUS,STAT_DL_VALUE,STAT_DL_DUP_VALUE,LIM_REP_VALUE,LIM_REP_DUP_VALUE,UNIT_CODE
19307,TA,2021-05-10 11:39:33.000000,2.185365,3.378640,1.193275,2.782003,42.89,Pass,12.5,12.5,10,15.0,MG_KG
19308,TA,2021-05-10 11:39:33.000000,0.801886,-0.746270,1.548156,0.027808,5567.32,Pass,12.5,12.5,10,15.0,MG_KG
19309,TA,2021-05-10 11:39:33.000000,1.260273,1.879069,0.618796,1.569671,39.42,Pass,12.5,12.5,10,15.0,MG_KG
19323,TA,2021-05-10 11:38:57.000000,-2.010870,-3.129188,1.118317,-2.570029,-43.51,Pass,12.5,12.5,10,15.0,MG_KG
19324,TA,2021-05-10 11:38:57.000000,-3.709679,-2.185186,1.524493,-2.947432,-51.72,Pass,12.5,12.5,10,15.0,MG_KG
19330,TE,2021-05-10 11:38:44.000000,5.205476,-4.130237,9.335714,0.537619,1736.49,Pass,25.0,25.0,10,15.0,MG_KG
19333,TE,2021-05-10 11:38:44.000000,3.537630,2.379626,1.158004,2.958628,39.14,Pass,25.0,25.0,10,15.0,MG_KG
19337,TE,2021-05-10 11:38:44.000000,-5.698118,2.995021,8.693140,-1.351549,-643.20,Pass,25.0,25.0,10,15.0,MG_KG
19339,TE,2021-05-10 11:38:44.000000,-2.273175,-1.864082,0.409093,-2.068629,-19.78,Pass,25.0,25.0,10,15.0,MG_KG
19343,TE,2021-05-10 11:38:44.000000,1.956517,1.148321,0.808196,1.552419,52.06,Pass,25.0,25.0,10,15.0,MG_KG


In [15]:
df_replicates_model.head(10)


,ANALYTE_CODE,ANALYSED_DATE,NUMERIC_FINAL_VALUE,PARENT_NUMERIC_FINAL_VALUE,ABS_DIFF,MEAN_CONC,RPD,PRECISION_STATUS,STAT_DL_VALUE,STAT_DL_DUP_VALUE,LIM_REP_VALUE,LIM_REP_DUP_VALUE,UNIT_CODE
2,PB,2021-11-22 16:37:52.000000,1492.268392,1385.198107,107.070285,1438.733250,7.44,Warning,1.250,1.250,10,15.0,MG_KG
3,NI,2021-11-22 16:37:33.000000,18.525452,16.334782,2.190670,17.430117,12.57,Pass,5.000,5.000,10,15.0,MG_KG
9,AG,2021-11-22 16:35:49.000000,14.901179,14.647905,0.253274,14.774542,1.71,Pass,0.050,0.050,10,15.0,MG_KG
14,CU,2021-11-22 16:35:49.000000,19641.928358,18891.143085,750.785274,19266.535721,3.90,Pass,5.000,5.000,10,15.0,MG_KG
15,ZN,2021-11-22 16:35:49.000000,2419.042385,2340.890959,78.151426,2379.966672,3.28,Pass,12.500,12.500,10,15.0,MG_KG
20,BE,2021-11-10 07:12:48.000000,0.220098,0.216041,0.004057,0.218070,1.86,Pass,0.250,0.250,10,15.0,MG_KG
21,BE,2021-11-10 07:12:48.000000,0.187723,0.156735,0.030988,0.172229,17.99,Pass,0.250,0.250,10,15.0,MG_KG
25,MO,2021-11-10 07:12:38.000000,358.962758,357.062846,1.899912,358.012802,0.53,Pass,0.125,0.125,10,15.0,MG_KG
29,MO,2021-11-10 07:12:38.000000,351.878831,359.480473,7.601642,355.679652,2.14,Pass,0.125,0.125,10,15.0,MG_KG
31,SB,2021-11-10 07:11:51.000000,161.816372,137.521696,24.294676,149.669034,16.23,IgnoreFailure,0.125,0.125,10,15.0,MG_KG


In [13]:
# Add precision statistics to duplicate dataset
df_duplicates["ABS_DIFF"] = abs(
    df_duplicates["NUMERIC_FINAL_VALUE"] - df_duplicates["PARENT_NUMERIC_FINAL_VALUE"]
)

df_duplicates["MEAN_CONC"] = (
    df_duplicates["NUMERIC_FINAL_VALUE"] + df_duplicates["PARENT_NUMERIC_FINAL_VALUE"]
) / 2

df_duplicates["RPD"] = np.where(
    df_duplicates["MEAN_CONC"] != 0,
    df_duplicates["ABS_DIFF"] / df_duplicates["MEAN_CONC"] * 100,
    np.nan
)

# Inspect the new columns
df_duplicates[
    [
        "NUMERIC_FINAL_VALUE",
        "PARENT_NUMERIC_FINAL_VALUE",
        "ABS_DIFF",
        "MEAN_CONC",
        "RPD"
    ]
].head()

df_duplicates


,NUMERIC_FINAL_VALUE,PARENT_NUMERIC_FINAL_VALUE,ANALYTE_CODE,STAT_DL_VALUE,STAT_DL_DUP_VALUE,LIM_REP_VALUE,LIM_REP_DUP_VALUE,UNIT_CODE,PRECISION_STATUS,ABS_DIFF,MEAN_CONC,RPD
19307,2.185365,3.378640,TA,12.50,12.50,10,15.0,MG_KG,Pass,1.193275,2.782003,42.892667
19308,0.801886,-0.746270,TA,12.50,12.50,10,15.0,MG_KG,Pass,1.548156,0.027808,5567.321671
19309,1.260273,1.879069,TA,12.50,12.50,10,15.0,MG_KG,Pass,0.618796,1.569671,39.422011
19323,-2.010870,-3.129188,TA,12.50,12.50,10,15.0,MG_KG,Pass,1.118317,-2.570029,-43.513797
19324,-3.709679,-2.185186,TA,12.50,12.50,10,15.0,MG_KG,Pass,1.524493,-2.947432,-51.722735
...,...,...,...,...,...,...,...,...,...,...,...,...
99403,-3.794613,-7.602763,U,50.00,50.00,10,15.0,MG_KG,Pass,3.808151,-5.698688,-66.825040
99588,18.575758,18.010363,LA,1.25,1.25,10,15.0,MG_KG,Pass,0.565395,18.293060,3.090762
99589,33.343434,32.424870,CE,12.50,12.50,10,15.0,MG_KG,Pass,0.918564,32.884152,2.793333
99590,77.414141,78.082902,SE,25.00,25.00,10,15.0,MG_KG,Pass,0.668760,77.748521,0.860158


## Part A — Unscaled Duplicate Precision Exploration

This section explores duplicate precision behaviour using the original, unscaled measurement values. The aim is to understand the natural distribution of duplicate differences before applying scaling or machine learning models.

In [ ]:
# Check the resulting feature dataset
print("Shape:", df_duplicates.shape)

df_duplicates[
    [
        "NUMERIC_FINAL_VALUE",
        "PARENT_NUMERIC_FINAL_VALUE",
        "ABS_DIFF",
        "RPD",
        "MEAN_VALUE",
        "LOG_MEAN"
    ]
].describe()


In [ ]:
df_duplicates_BI = df_duplicates[df_duplicates["ANALYTE_CODE"] == "BI"].copy()

# Export to CSV
df_duplicates_BI.to_csv("../data/processed/duplicates_BI_only.csv", index=False)


In [ ]:
df_duplicates_BI["PRECISION_STATUS"].value_counts()

In [ ]:
df_bi[
    [
        "NUMERIC_FINAL_VALUE",
        "PARENT_NUMERIC_FINAL_VALUE",
        "ABS_DIFF",
        "RPD",
        "MEAN_VALUE",
        "LOG_MEAN"
    ]]


In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter(
    df_bi["PARENT_NUMERIC_FINAL_VALUE"],
    df_bi["NUMERIC_FINAL_VALUE"],
    alpha=0.6
)

min_val = min(df_bi["PARENT_NUMERIC_FINAL_VALUE"].min(), df_bi["NUMERIC_FINAL_VALUE"].min())
max_val = max(df_bi["PARENT_NUMERIC_FINAL_VALUE"].max(), df_bi["NUMERIC_FINAL_VALUE"].max())

plt.plot([min_val, max_val], [min_val, max_val])

plt.xlabel("Parent / Original BI Value")
plt.ylabel("Duplicate BI Value")
plt.title("Bismuth Duplicate vs Original Values")
plt.show()


In [ ]:
features = df_bi[["RPD", "LOG_MEAN"]].replace([np.inf, -np.inf], np.nan).dropna()

print("Original BI rows:", len(df_bi))
print("Rows available for modelling:", len(features))
print("Rows excluded:", len(df_bi) - len(features))


In [ ]:
plt.figure(figsize=(7, 5))

plt.hist(df_bi["RPD"].dropna(), bins=30)

plt.xlabel("Relative Percent Difference (RPD)")
plt.ylabel("Frequency")
plt.title("Distribution of BI Duplicate Precision Error")
plt.show()


In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(features)

X.shape


In [ ]:
kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

df_bi["CLUSTER"] = np.nan
df_bi.loc[features.index, "CLUSTER"] = kmeans.fit_predict(X)

df_bi["CLUSTER"].value_counts(dropna=False)


In [ ]:
df_bi.groupby("CLUSTER")["PRECISION_STATUS"].value_counts()


In [ ]:
df_bi.groupby("CLUSTER")[["RPD", "MEAN_VALUE", "LOG_MEAN", "ABS_DIFF"]].agg(
    ["count", "mean", "median", "min", "max"]
)


In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter(
    df_bi.loc[features.index, "LOG_MEAN"],
    df_bi.loc[features.index, "RPD"],
    c=df_bi.loc[features.index, "CLUSTER"],
    alpha=0.7
)

plt.xlabel("Log Mean BI Concentration")
plt.ylabel("Relative Percent Difference (RPD)")
plt.title("KMeans Clusters for BI Duplicate Measurements")
plt.show()


In [ ]:
iso = IsolationForest(
    contamination=0.05,
    random_state=42
)

df_bi["ANOMALY_SCORE"] = np.nan
df_bi.loc[features.index, "ANOMALY_SCORE"] = iso.fit_predict(X)

df_bi["ANOMALY_SCORE"].value_counts(dropna=False)


In [ ]:
df_bi.groupby("ANOMALY_SCORE")["PRECISION_STATUS"].value_counts()


In [ ]:
df_bi[df_bi["ANOMALY_SCORE"] == -1][[
    "JOB_CODE",
    "ANALYTE_CODE",
    "NUMERIC_FINAL_VALUE",
    "PARENT_NUMERIC_FINAL_VALUE",
    "ABS_DIFF",
    "RPD",
    "MEAN_VALUE",
    "LOG_MEAN",
    "PRECISION_STATUS",
    "CLUSTER",
    "ANOMALY_SCORE"
]].sort_values("RPD", ascending=False)


In [ ]:
df_bi.to_csv(
    "../data/processed/duplicates_BI_enriched.csv",
    index=False
)
